# 🌐 Import & Inspect SPOKE‑GeneLab Knowledge Graph in Neo4j

This notebook bulk‑imports the CSV node and relationship exports from the SPOKE‑GeneLab pipeline into a Neo4j graph database, then runs exploratory queries to validate schema, metadata, counts, and full‑text search functionality. It supports downloading into a Neo4j Desktop database or Neo4j Enterprise instance (see cell [3])

**Start the `spoke-genelab` Graph DBMS before you run this notebook!**

Author: Peter W. Rose, UC San Diego (pwrose.ucsd@gmail.com)

In [1]:
import os
import pandas as pd
from py2neo import Graph
import neo4j_utils
import neo4j_bulk_importer

In [2]:
pd.set_option('display.max_rows', None)  # Shows all rows
pd.set_option('display.max_columns', None) # Shows all columns
pd.set_option('display.max_colwidth', None)  # Shows full content of each cell

### Import the Knowledge Graph
CSV data and metadata files are uploaded into the Neo4j Graph database from the [kg](https://github.com/BaranziniLab/spoke_genelab/tree/main/kg) directory using the [kg-import](https://github.com/sbl-sdsc/kg-import) bulk upload scripts. For a description of the data organization and the specification of metadata [see](https://github.com/sbl-sdsc/kg-import/blob/main/README.md).

In [3]:
neo4j_bulk_importer.import_from_csv_to_neo4j_desktop(verbose=True) # Use for Neo4j Desktop
# neo4j_bulk_importer.import_from_csv_to_neo4j_enterprise(verbose=True) # Use for Neo4j Enterprise instance

drop_database: '/Users/peter/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-a7d8aa2e-bcae-4309-89c9-cf934cc8ee5e/bin/cypher-shell' -d system -u neo4j -p neo4jdemo 'DROP DATABASE `spoke-genelab-v0.3.1` IF EXISTS;'



Executing:   0%|          | 0/85 [00:00<?, ?cell/s]

run_bulk_import: export NEO4J_ADMIN_JAVA_OPTS='-Xms16g -Xmx20g -XX:+UseG1GC'; cd '/Users/peter/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-a7d8aa2e-bcae-4309-89c9-cf934cc8ee5e/import'; '/Users/peter/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-a7d8aa2e-bcae-4309-89c9-cf934cc8ee5e/bin/neo4j-admin' database import full spoke-genelab-v0.3.1 --overwrite-destination --skip-bad-relationships --skip-duplicate-nodes --max-off-heap-memory=8g --array-delimiter='|' @args.txt
Starting to import, output will be saved to: /Users/peter/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-a7d8aa2e-bcae-4309-89c9-cf934cc8ee5e/logs/neo4j-admin-import-2026-03-10.18.36.10.log
Neo4j version: 2025.12.1
Importing the contents of these files into /Users/peter/Library/Application Support/neo4j-desktop/Application/Data/dbmss/dbms-a7d8aa2e-bcae-4309-89c9-cf934cc8ee5e/data/databases/spoke-genelab-v0.3.1:
Nodes:
  [Assay]:
  /Users/peter/Library/A

### Connect to the local Neo4j Graph database

In [4]:
database = os.environ.get("NEO4J_DATABASE")
username = os.environ.get("NEO4J_USERNAME")
password = os.environ.get("NEO4J_PASSWORD")
stylesheet = os.environ.get("NEO4J_STYLESHEET")

graph = Graph("bolt://localhost:7687", name=database, user=username, password=password)

## Metadata <a class="anchor" id="Metadata"></a>

### Node metadata
spoke_genelab KG is a self-describing KG. The MetaNodes and MetaRelationships define the structure of the KG and the properties of nodes and relationships. The query below lists the nodes and their properties.

In [5]:
query = """
MATCH (n:MetaNode) RETURN n;
"""
df = graph.run(query).to_data_frame()
metadata = df["n"].tolist()
metadata = pd.DataFrame(metadata)
metadata.fillna("", inplace=True)
metadata

,nodeName,identifier,factor_space_2,differential_analysis_method,factor_space_1,material_name_1,technology,material_name_2,measurement,factors_2,material_2,factors_1,material_1,name,material_id_1,material_id_2,symbol,organism,taxonomy,dist_to_feature,in_exon,in_promoter,chromosome,start,in_intron,end,end_date,flight_program,space_program,start_date,host_strain,project_type,description,host_organism,project_title,host_taxonomy
0,Anatomy,UBERON Ontology ID (string),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,Assay,Unique assay identifier combining GeneLab dataset accession and MD5 hash of metadata (links assay to dataset and ensures version integrity) (string),"Secondary experimental grouping for samples used in the assay, indicating whether it was maintained under control conditions: Ground Control, Basal Control, or Vivarium Control (string)","The statistical method used to test for differential signal between conditions. Applies to: Differential expression, methylation, abundance (string)","Primary experimental grouping for samples used the in assay, indicating whether it was exposed to Space Flight or other conditions (string)","Preferred ontology-based name for the first material (e.g., UBERON term) (string)","Platform or method used to generate assay data (e.g., sequencing platform, mass spectrometry, imaging) (string)","Preferred ontology-based name for the second material (e.g., UBERON term) (string)","Type of data produced by the assay, describing the focus of the analysis (string)","Secondary experimental factor(s) for this assay, typically the comparative or control group(s) in analysis connecting this assay to differential conditions (e.g., Ground Control). (string[])","Biological material analyzed in the second assay group (e.g., tissue, cell type, organ) (string)","Primary experimental factor(s) applied at the assay level (independent variables intentionally varied by the investigator); often corresponds to study-level design (e.g., Space Flight). Additional factors (e.g., dose, time point, sex, strain) may also be included. (string[])","Biological material analyzed in the first assay group (e.g., tissue, cell type, organ) (string)",Human-readable assay title detailing the biological measurement performed (string),"Controlled ontology identifier for the first material (e.g., UBERON code) (string)","Controlled ontology identifier for the second material (e.g., UBERON code) (string)",,,,,,,,,,,,,,,,,,,,
2,CellType,Cell Ontology ID (string),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
3,Gene,Gene ENTREZID (string),,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
4,MGene,Gene ENTREZID for model organism (string),,,,,,,,,,,,Gene name for model organism (string),,,Gene symbol for model organism (string),NCBI scientifc name for the model organism (string),NCBI taxonomy id for the model organism (string),,,,,,,,,,,,,,,,,
5,MethylationRegion,Identifier for base pair range (string),,,,,,,,,,,,Name of methylation region (string),,,,,,Distance to feature (int),Indicates whether the region overlaps with an exon (boolean),Indicates whether the region overlaps with the promoter (boolean),Name or number of the chromosome (string),Starting base pair position (int),Indicates whether the region overlaps with an intron (boolean),Ending base pair position (int),,,,,,,,,,
6,Mission,Identifier for mission (string),,,,,,,,,,,,Name of the study (string),,,,,,,,,,,,,End date of mission (date),Type of flight program (string),Sponsor of the mission (string),Start date of mission (date),,,,,,
7,Organism,NCBI taxonomy ID (string),,,,,,,,,,,,NCBI scientific organism name (string),,,,,,,,,,,,,,,,,,,,,,
8,Study,NASA Open Science Data Repository ID (string),,,,,,,,,,,,Name of the study (string),,,,NCBI scientific organism name (string),NCBI taxonomy ID (string),,,,,,,,,,,,"Specific genetic background, cultivar, or strain identifier of the host organism, distinguishing sub-populations or lines used in the study to control for genetic variation (string)",Type of project (string),Desc

### Metagraph <a class="anchor" id="Metagraph"></a>
The metagraph shows the node labels and relationship types of the KG. Click on a node to display the node metadata.

In [6]:
query = """
MATCH p=(:MetaNode)-->(:MetaNode) RETURN p
"""
subgraph1 = graph.run(query).to_subgraph()

In [7]:
# If the layout isn’t ideal, rerun this cell.
# Use the mouse to drag nodes, pan or zoom the canvas, and adjust positions.
# Click a node to view its properties.
widget1 = neo4j_utils.draw_graph(subgraph1, stylesheet)
widget1.layout.height = "1014px"
widget1.set_layout(name="cola", padding=00, nodeSpacing=65, nodeDimensionsIncludeLabels=True, unconstrIter=15000)
display(widget1)

CytoscapeWidget(cytoscape_layout={'name': 'cola', 'padding': 0, 'nodeSpacing': 65, 'nodeDimensionsIncludeLabel…

### Number of Nodes

In [8]:
query = """
MATCH (n) RETURN COUNT(n);
"""
print(f"Total number of nodes: {graph.evaluate(query)}")

Total number of nodes: 190252


In [9]:
query = """
MATCH (n) RETURN labels(n)[0] AS Node, COUNT(n) AS Count
ORDER BY Count DESC;
"""
graph.run(query).to_data_frame()

,Node,Count
0,MGene,138736
1,Gene,30009
2,Assay,13146
3,MethylationRegion,8003
4,Study,212
5,Organism,46
6,Anatomy,42
7,Mission,42
8,MetaNode,9
9,CellType,7


### Number of relationships

In [10]:
query = """
MATCH (n1)-[r]->(n2)
WITH TYPE(r)        AS Relationship,
     count(r)       AS Count,
     LABELS(n1)[0]  AS Node1,
     LABELS(n2)[0]  AS Node2
RETURN Node1, Relationship, Node2, Count
ORDER BY Count DESC;
"""
graph.run(query).to_data_frame()

,Node1,Relationship,Node2,Count
0,Assay,MEASURED_DIFFERENTIAL_EXPRESSION_ASmMG,MGene,50208066
1,MGene,IS_ORTHOLOG_MGiG,Gene,121153
2,Assay,MEASURED_DIFFERENTIAL_METHYLATION_ASmMR,MethylationRegion,27454
3,Study,PERFORMED_SpAS,Assay,13146
4,MGene,METHYLATED_IN_MGmMR,MethylationRegion,8270
5,Assay,INVESTIGATED_ASiA,Anatomy,7214
6,Assay,INVESTIGATED_ASiCT,CellType,1240
7,Assay,MEASURED_DIFFERENTIAL_ABUNDANCE_ASmO,Organism,193
8,Mission,CONDUCTED_MIcS,Study,127
9,MetaNode,MetaRelationship,MetaNode,9


## Fulltext Search <a class="anchor" id="Fulltext Search"></a>

### Full text query by keyword or phrase
A full text query returns all nodes that match the text query ([Query Syntax](https://lucene.apache.org/core/5_5_0/queryparser/org/apache/lucene/queryparser/classic/package-summary.html#Overview)). For exact matches, enclose the phrase in double quotes, e.g., ```'"eye"'```.

[Learn more about full-text searches.](https://graphaware.com/neo4j/2019/01/11/neo4j-full-text-search-deep-dive.html)

In [11]:
phrase = '"eye"'
top_n = 5 # show top n hits

In [12]:
query = """
CALL db.index.fulltext.queryNodes("fulltext", $phrase) YIELD node, score
RETURN node.identifier AS identifier, LABELS(node)[0] AS type, node;
"""
graph.run(query, phrase=phrase).to_data_frame().head(top_n)

,identifier,type,node
0,OSD-162-66f0061db09331915745ce1f52d2ed2b,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'DESeq2', 'factor_space_1': 'Basal Control', 'identifier': 'OSD-162-66f0061db09331915745ce1f52d2ed2b', 'material_name_1': 'eye', 'technology': 'RNA Sequencing (RNA-Seq)', 'material_name_2': 'eye', 'measurement': 'transcription profiling', 'factors_2': ['Ground Control'], 'material_2': 'eye', 'factors_1': ['Basal Control'], 'material_1': 'eye', 'name': 'OSD-162_transcription-profiling_rna-sequencing-(rna-seq)_illumina', 'material_id_1': 'UBERON:0000970', 'material_id_2': 'UBERON:0000970'}"
1,OSD-162-87eb8168736ada740a058a0da8705990,Assay,"{'factor_space_2': 'Space Flight', 'differential_analysis_method': 'DESeq2', 'factor_space_1': 'Basal Control', 'identifier': 'OSD-162-87eb8168736ada740a058a0da8705990', 'material_name_1': 'eye', 'technology': 'RNA Sequencing (RNA-Seq)', 'material_name_2': 'eye', 'measurement': 'transcription profiling', 'factors_2': ['Space Flight'], 'material_2': 'eye', 'factors_1': ['Basal Control'], 'material_1': 'eye', 'name': 'OSD-162_transcription-profiling_rna-sequencing-(rna-seq)_illumina', 'material_id_1': 'UBERON:0000970', 'material_id_2': 'UBERON:0000970'}"
2,OSD-162-0beedae29327a0e882373736c0bfcfcd,Assay,"{'factor_space_2': 'Space Flight', 'differential_analysis_method': 'DESeq2', 'factor_space_1': 'Ground Control', 'identifier': 'OSD-162-0beedae29327a0e882373736c0bfcfcd', 'material_name_1': 'eye', 'technology': 'RNA Sequencing (RNA-Seq)', 'material_name_2': 'eye', 'measurement': 'transcription profiling', 'factors_2': ['Space Flight'], 'material_2': 'eye', 'factors_1': ['Ground Control'], 'material_1': 'eye', 'name': 'OSD-162_transcription-profiling_rna-sequencing-(rna-seq)_illumina', 'material_id_1': 'UBERON:0000970', 'material_id_2': 'UBERON:0000970'}"
3,OSD-162-d11d245372149548d119d60e948a394d,Assay,"{'factor_space_2': 'Basal Control', 'differential_analysis_method': 'DESeq2', 'factor_space_1': 'Ground Control', 'identifier': 'OSD-162-d11d245372149548d119d60e948a394d', 'material_name_1': 'eye', 'technology': 'RNA Sequencing (RNA-Seq)', 'material_name_2': 'eye', 'measurement': 'transcription profiling', 'factors_2': ['Basal Control'], 'material_2': 'eye', 'factors_1': ['Ground Control'], 'material_1': 'eye', 'name': 'OSD-162_transcription-profiling_rna-sequencing-(rna-seq)_illumina', 'material_id_1': 'UBERON:0000970', 'material_id_2': 'UBERON:0000970'}"
4,OSD-162-2d639271ebaf8e5a0d64aed923092239,Assay,"{'factor_space_2': 'Basal Control', 'differential_analysis_method': 'DESeq2', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-162-2d639271ebaf8e5a0d64aed923092239', 'material_name_1': 'eye', 'technology': 'RNA Sequencing (RNA-Seq)', 'material_name_2': 'eye', 'measurement': 'transcription profiling', 'factors_2': ['Basal Control'], 'material_2': 'eye', 'factors_1': ['Space Flight'], 'material_1': 'eye', 'name': 'OSD-162_transcription-profiling_rna-sequencing-(rna-seq)_illumina', 'material_id_1': 'UBERON:0000970', 'material_id_2': 'UBERON:0000970'}"


### Full text query using boolean operators
The full text query supports a variety of query types, including fuzzy, proximity, and range queries, as well as boolean operators ([Query Syntax](https://lucene.apache.org/core/5_5_0/queryparser/org/apache/lucene/queryparser/classic/package-summary.html#Overview)). The following example uses a query with an ```AND``` operator.

In [13]:
phrase = 'liver AND "DNA methylation profiling"'

In [14]:
query = """
CALL db.index.fulltext.queryNodes("fulltext", $phrase) YIELD node, score
RETURN node.identifier AS identifier, LABELS(node)[0] AS type, node, score
ORDER BY type;
"""
graph.run(query, phrase=phrase).to_data_frame()

,identifier,type,node,score
0,OSD-47-cd63149def82bdabff6c53b156b53161,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Basal Control', 'identifier': 'OSD-47-cd63149def82bdabff6c53b156b53161', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Ground Control'], 'material_2': 'Liver', 'factors_1': ['Basal Control'], 'material_1': 'Liver', 'name': 'OSD-47_dna-methylation-profiling_whole-genome-bisulfite-sequencing_illumina', 'material_id_1': 'UBERON:0002107', 'material_id_2': 'UBERON:0002107'}",8.541791
1,OSD-48-d179e7f3043445de5c21d8c3b3a8aa37,Assay,"{'factor_space_2': 'Space Flight', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-48-d179e7f3043445de5c21d8c3b3a8aa37', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Space Flight', 'Carcass'], 'material_2': 'Liver', 'factors_1': ['Space Flight', 'Upon euthanasia'], 'material_1': 'Liver', 'name': 'OSD-48_dna-methylation-profiling_whole-genome-bisulfite-sequencing_Illumina', 'material_id_1': 'UBERON:0002107', 'material_id_2': 'UBERON:0002107'}",8.541791
2,OSD-48-8802deba3d00cd3418c2ac977edd7e9e,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-48-8802deba3d00cd3418c2ac977edd7e9e', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Ground Control', 'Upon euthanasia'], 'material_2': 'Liver', 'factors_1': ['Space Flight', 'Upon euthanasia'], 'material_1': 'Liver', 'name': 'OSD-48_dna-methylation-profiling_whole-genome-bisulfite-sequencing_Illumina', 'material_id_1': 'UBERON:0002107', 'material_id_2': 'UBERON:0002107'}",8.541791
3,OSD-48-e2589e81638e7a17414474f5f359e566,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-48-e2589e81638e7a17414474f5f359e566', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Ground Control', 'Upon euthanasia'], 'material_2': 'Liver', 'factors_1': ['Space Flight', 'Carcass'], 'material_1': 'Liver', 'name': 'OSD-48_dna-methylation-profiling_whole-genome-bisulfite-sequencing_Illumina', 'material_id_1': 'UBERON:0002107', 'material_id_2': 'UBERON:0002107'}",8.541791
4,OSD-48-fe70005646d5fdcbcbfeca5ec864eef0,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-48-fe70005646d5fdcbcbfeca5ec864eef0', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Ground Control', 'Carcass'], 'material_2': 'Liver', 'factors_1': ['Space Flight', 'Upon euthanasia'], 'material_1': 'Liver', 'name': 'OSD-48_dna-methylation-profiling_whole-genome-bisulfite-sequencing_Illumina', 'material_id_1': 'UBERON:0002107', 'material_id_2': 'UBERON:0002107'}",8.541791
5,OSD-48-4dc1fe3d0a90442bfef1111b48ce247b,Assay,"{'factor_space_2': 'Ground Control', 'differential_analysis_method': 'methylKit', 'factor_space_1': 'Space Flight', 'identifier': 'OSD-48-4dc1fe3d0a90442bfef1111b48ce247b', 'material_name_1': 'liver', 'technology': 'Whole Genome Bisulfite Sequencing', 'material_name_2': 'liver', 'measurement': 'DNA methylation profiling', 'factors_2': ['Ground Control', 'Carcass'], 'material_2': 'Liver', 'factors_1': ['Space Flight', 'Carcass'], 'material_1': 'Liver', 'name': 'OSD-48_dna-methylation-profiling_whole-genome-bisulfite-sequencing_Illumina', 'mat